# Is our loss the weak link? SURE against R2R at equal budget, on a denoiser small enough for CPU

The paper's adaptation (§4.3) uses a Recorrupted-to-Recorrupted loss. Its closest prior art, GainTuning (NeurIPS
2021), tunes the same kind of parameter — per-channel gains on one test image — with a **SURE** loss, and reports that
it "does not require early stopping". We never compared the two.

Running that comparison on ADM needs GPU hours we do not have. This kernel runs it on a denoiser that fits CPU:
**DnCNN-gray, blind** (KAIR, 0.56 M parameters, pretrained on AWGN). The question is narrow and portable:
*at equal parameter set, equal step budget and equal crops, which loss gains more, and which one harms?*

Arms, all starting from the same pretrained weights and resetting to them between cases:

| arm | loss | target |
|---|---|---|
| `oracle` | MSE against the clean image | upper bound on what these parameters can do |
| `r2r` | MSE(f(y + ασz), y − σz/α), α = 0.5 | self-supervised, our method's loss |
| `sure` | mean((f(y) − y)²) + 2σ²·div f, divergence by Hutchinson | self-supervised, GainTuning's loss |

Adapted parameters: one gain per output channel of every convolution, identical for all three arms.

Cases: BSD68 (30 images) and 512² centre crops of Urban100 (20 images), at σ = 25 and 50 — the split where our
pipeline gains on large images and loses on small ones.

Pre-registered reading (fixed before the run):
* **`sure` beats `r2r`** if its mean gain is higher by at least 0.02 dB on both sets and its harm rate (images worse
  by > 0.05 dB) is no higher;
* **`r2r` beats `sure`** under the reverse;
* **no difference** if the two are within 0.02 dB on both sets, in which case our loss is not the weak link and the
  paper's contribution stays the certificate and predictor rather than the objective.
Kaggle: CPU, internet on.

In [ ]:
import io, math, os, re, time, urllib.request, zipfile, zlib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image

torch.set_num_threads(os.cpu_count())
ALPHA, STEPS, CROP_N, CROP, LR = 0.5, 60, 2, 96, 1e-3
EPS_REL = 1e-3
SIGMAS = [25, 50]
T_LIMIT_H = 10.0
CACHE, OUT = "/tmp/cache", "outputs"
for d in (CACHE, OUT):
    os.makedirs(d, exist_ok=True)
T0 = time.time()
print(f"{os.cpu_count()} CPUs, {STEPS} steps, {CROP_N} x {CROP}² crops")


def download(url, path):
    if os.path.exists(path) and os.path.getsize(path) > 0:
        return path
    for a in range(5):
        try:
            with urllib.request.urlopen(url, timeout=300) as r, open(path, "wb") as f:
                while chunk := r.read(1 << 23):
                    f.write(chunk)
            return path
        except Exception as e:
            print(f"  retry {a+1}: {e}", flush=True)
            time.sleep(5 * (a + 1))
    raise RuntimeError(url)

## DnCNN-gray, the KAIR architecture (17 layers, BN + ReLU) and its released blind-denoiser weights

In [ ]:
def dncnn(nc=64, depth=20):
    """KAIR's DnCNN-V1 for blind gray denoising, built to match its checkpoint exactly: `depth` convolutions of 3x3,
    every one with a bias, a ReLU after each of the first `depth - 1`, and no normalisation. The released
    `dncnn_gray_blind.pth` holds 20 convs at state-dict indices 0, 2, …, 38, which is what fixes these numbers rather
    than a guess."""
    layers = []
    for i in range(depth - 1):
        layers += [nn.Conv2d(1 if i == 0 else nc, nc, 3, 1, 1, bias=True), nn.ReLU(inplace=True)]
    layers += [nn.Conv2d(nc, 1, 3, 1, 1, bias=True)]
    return nn.Sequential(*layers)


W = download("https://github.com/cszn/KAIR/releases/download/v1.0/dncnn_gray_blind.pth", f"{CACHE}/dncnn_gray_blind.pth")
state = torch.load(W, map_location="cpu", weights_only=True)
if "state_dict" in state:
    state = state["state_dict"]
# KAIR checkpoints are saved through DataParallel wrappers: strip any "module." / "model." prefixes.
def strip(k):
    while True:
        for pre in ("module.", "model."):
            if k.startswith(pre):
                k = k[len(pre):]
                break
        else:
            return k
state = {strip(k): v for k, v in state.items()}
BASE = dncnn()
missing, unexpected = BASE.load_state_dict(state, strict=False)
missing = [k for k in missing if "num_batches_tracked" not in k]
print("missing:", missing[:6], "| unexpected:", unexpected[:6])
assert not missing and not unexpected, "DnCNN weights did not load: fix the prefix handling before running"
BASE.eval().requires_grad_(False)
print("params:", sum(p.numel() for p in BASE.parameters()) / 1e6, "M")


def denoise(y01):
    """The released checkpoint takes [0,1] inputs and returns the *residual*: clean = y - residual.

    Established empirically, on one image, over four candidate conventions: feeding [0,1] and subtracting the output
    scores 28.22 / 25.85 / 23.16 dB on BSD68 test001 at σ = 15 / 25 / 50, while [0,255] input, [0,255]-input-with-
    subtraction and [0,1]-without-subtraction score 7.3–22.9 dB. The output's standard deviation at σ 25 is 0.090
    against a noise standard deviation of 0.098, which is what a residual estimator should look like.

    This sits about 3 dB below the published blind gray numbers (29.23 / 29.16 / 26.23) on our protocol, a gap we did
    not chase because it does not affect what this kernel measures: both arms start from these same weights and the
    endpoint is the *gain* over the unadapted baseline.
    """
    return y01 - BASE(y01)

In [ ]:
def make_adaptable(net):
    """One gain per output channel of every Conv2d; returns the parameter list. Weighted so the pretrained behaviour is
    exactly reproduced at initialisation (gain = 1)."""
    gains = []
    for m in net.modules():
        if isinstance(m, nn.Conv2d):
            g = nn.Parameter(torch.ones(m.out_channels))
            m.register_forward_hook(lambda mod, inp, out, g=g: out * g.view(1, -1, 1, 1))
            gains.append(g)
    return gains


GAINS = make_adaptable(BASE)
print("adapted parameters:", sum(g.numel() for g in GAINS))


def reset():
    with torch.no_grad():
        for g in GAINS:
            g.fill_(1.0)

## Data

In [ ]:
RAW = "https://raw.githubusercontent.com/cszn/DnCNN/master/testsets/{}/{}"
BSD = [(f"test{i:03d}.png", np.asarray(Image.open(download(RAW.format("BSD68", f"test{i:03d}.png"), f"{CACHE}/b{i}.png")).convert("L")))
       for i in range(1, 25)]
zs = download("https://github.com/Li-Tong-621/DMID/releases/download/v1.0/Datasets.zip", f"{CACHE}/Datasets.zip")
with zipfile.ZipFile(zs) as z:
    names = sorted([n for n in z.namelist() if n.startswith("Datasets/Gaussion/Urban100/") and not n.endswith("/")],
                   key=lambda s: [int(t) if t.isdigit() else t for t in re.split(r"(\d+)", s)])[:12]
    URB = [(n.split("/")[-1], np.asarray(Image.open(io.BytesIO(z.read(n))).convert("L"))) for n in names]
print("BSD68", len(BSD), "| Urban100", len(URB))


def noisy(clean_u8, sigma, seed):
    """DMID's protocol: sample std exactly σ/255, unclipped."""
    x = torch.from_numpy(np.float32(clean_u8[None, None]) / 255.0)
    g = torch.Generator().manual_seed(seed)
    n = torch.randn(x.shape, generator=g)
    return x + (sigma / 255.0) * n / n.std(unbiased=False)


def centre_crop(img, size):
    h, w = img.shape
    i, j = max(0, (h - size) // 2), max(0, (w - size) // 2)
    return img[i:i + size, j:j + size]


def psnr(x, clean_u8):
    a = (x.clamp(0, 1) * 255).round().double()
    b = torch.from_numpy(np.float64(clean_u8)[None, None])
    return 10 * math.log10(255 ** 2 / (a - b).pow(2).mean().item())

In [ ]:
def crops_pair(y, clean, gen):
    """CROP_N crops with identical indices in the noisy and the clean image."""
    h, w = y.shape[-2:]
    if h <= CROP or w <= CROP:
        return y.repeat(CROP_N, 1, 1, 1), clean.repeat(CROP_N, 1, 1, 1)
    ys, cs = [], []
    for _ in range(CROP_N):
        i = int(torch.randint(0, h - CROP + 1, (1,), generator=gen))
        j = int(torch.randint(0, w - CROP + 1, (1,), generator=gen))
        ys.append(y[..., i:i + CROP, j:j + CROP])
        cs.append(clean[..., i:i + CROP, j:j + CROP])
    return torch.cat(ys, 0), torch.cat(cs, 0)


def adapt(y, clean, sigma, arm, seed):
    """Run one arm from the pretrained weights; return the full-image estimate and its PSNR."""
    s = sigma / 255.0
    gen = torch.Generator().manual_seed(seed)
    opt = torch.optim.Adam(GAINS, lr=LR)
    cl = torch.from_numpy(np.float32(clean[None, None]) / 255.0)
    for _ in range(STEPS):
        c, cc = crops_pair(y, cl, gen)
        if arm == "oracle":
            loss = F.mse_loss(denoise(c), cc)
        elif arm == "r2r":
            z = torch.randn(c.shape, generator=gen)
            loss = F.mse_loss(denoise(c + ALPHA * s * z), c - s * z / ALPHA)
        elif arm == "sure":
            eps = EPS_REL * max(s, 1e-3)
            b = torch.randint(0, 2, c.shape, generator=gen).float() * 2 - 1
            f0 = denoise(c)
            f1 = denoise(c + eps * b)
            div = (b * (f1 - f0) / eps).mean()
            loss = ((f0 - c) ** 2).mean() + 2 * s ** 2 * div
        else:
            raise ValueError(arm)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()
    with torch.no_grad():
        est = denoise(y)
    reset()
    return est, psnr(est, clean)

In [ ]:
# Subset sweep: does the harm shrink when fewer parameters are fitted?
SUBSETS = {"100%": 1.0, "50%": 0.5, "25%": 0.25, "10%": 0.10}


def subset_gains(frac):
    """A fixed random subset of the gains, shared across cases, so only the count changes between arms."""
    n = sum(g.numel() for g in GAINS)
    keep = int(round(frac * n))
    rng = np.random.default_rng(0)
    mask = np.zeros(n, bool)
    mask[rng.choice(n, keep, replace=False)] = True
    out, k = [], 0
    for g in GAINS:
        m = mask[k:k + g.numel()]
        k += g.numel()
        if m.any():
            out.append((g, m))
    return out


def adapt_subset(y, clean, sigma, active, seed):
    s = sigma / 255.0
    gen = torch.Generator().manual_seed(seed)
    params = [g for g, _ in active]
    opt = torch.optim.Adam(params, lr=LR)
    cl = torch.from_numpy(np.float32(clean[None, None]) / 255.0)
    for _ in range(STEPS):
        c, _ = crops_pair(y, cl, gen)
        z = torch.randn(c.shape, generator=gen)
        loss = F.mse_loss(denoise(c + ALPHA * s * z), c - s * z / ALPHA)
        opt.zero_grad(set_to_none=True)
        loss.backward()
        for g, m in active:                    # zero the gradients of the held-out gains
            if g.grad is not None:
                g.grad[~m] = 0
        opt.step()
    with torch.no_grad():
        est = denoise(y)
    reset()
    return psnr(est, clean)

In [ ]:
CASES = [("BSD68", n, img) for n, img in BSD]
CASES += [("Urban100", n, centre_crop(img, 512)) for n, img in URB]
rows = []
t_start = time.time()
for ds, name, clean in CASES:
    for sigma in (25, 50):
        if time.time() - t_start > T_LIMIT_H * 3600:
            break
        y = noisy(clean, sigma, seed=zlib.crc32(f"{ds}/{name}".encode()) % 10_000 + 1000 * sigma)
        reset()
        with torch.no_grad():
            base = psnr(denoise(y), clean)
        assert base > 18, f"pretrained DnCNN scored {base:.2f} dB: weights are not loaded"
        for label, frac in SUBSETS.items():
            reset()
            t0 = time.time()
            p = adapt_subset(y, clean, sigma, subset_gains(frac), seed=7_000 + 97 * sigma)
            rows.append(dict(dataset=ds, image=name, sigma=sigma, subset=label, frac=frac, psnr=p, base=base,
                             gain=p - base, seconds=time.time() - t0))
        print(f"  {ds} {name} σ{sigma}: base {base:.3f} | " +
              " ".join(f"{r['subset']} {r['gain']:+.3f}" for r in rows[-len(SUBSETS):]), flush=True)
        pd.DataFrame(rows).to_csv(f"{OUT}/capacity_sweep.csv", index=False)

R = pd.DataFrame(rows)
R.to_csv(f"{OUT}/capacity_sweep.csv", index=False)
S = (R.pivot_table(index=["dataset", "sigma", "image"], columns="subset", values="gain")
     .groupby(level=["dataset", "sigma"]).agg(["mean"])
     .round(3))
S.columns = [c[1] for c in S.columns]
print("\ngain by parameter subset")
print(S.to_string())
worse = R[R.gain < -0.05].groupby(["dataset", "sigma", "subset"]).size().unstack(fill_value=0)
tot = R.groupby(["dataset", "sigma", "subset"]).size().unstack()
print("\nshare of images harmed (> 0.05 dB worse)")
print((worse / tot).round(2).fillna(0).to_string())
R.to_csv(f"{OUT}/capacity_sweep.csv", index=False)
print(f"done in {(time.time() - T0) / 60:.1f} min, {len(R)} rows")